# ¿Cuántos trazos bastan? · Notebook principal

Reconocimiento temprano de bocetos con CNN en Quick, Draw! (CC421, UNI, 2026-II).

Este notebook ejecuta **todo el proyecto** en orden con *Entorno de ejecución → Ejecutar todas*. Cada etapa guarda lo que produce en `trabajo/` (dentro de Drive) y, si ya existe, lo reutiliza: si Colab corta la sesión, basta volver a ejecutar todo y sigue donde quedó.

| Etapa | Qué hace | Responsable |
|---|---|---|
| 1 | Configuración y Drive | Elias |
| 2 | Datos, particiones y rasterización de todos los prefijos | Elias y Jorly |
| 3 | Búsqueda de hiperparámetros de M1 y M2 | Cristian |
| 4 | Entrenamiento de M1, M2 (3 semillas) y M2-P (3 semillas) | Cristian |
| 5 | Evaluación en validación: métricas, H1, H2, H4 y H5, figuras | Omar y Farid |
| 6 | Latencia | Omar |
| 7 | **Única evaluación en prueba** (semana 13, bloqueada por defecto) | Omar |
| 8 | Duración de los dibujos (opcional) | Elias |

La explicación de cada módulo está en la carpeta `docs/`.

**Antes de empezar:** usen un entorno con GPU (*Entorno de ejecución → Cambiar tipo de entorno → T4 GPU*).

In [ ]:
# 1. Configuración ---------------------------------------------------------------
import os, sys
from pathlib import Path

# True: prueba de punta a punta en minutos (sus números NO sirven para el paper).
# False: la corrida completa del proyecto.
MODO_RAPIDO = False
if os.environ.get("QD_MODO_RAPIDO") == "1":   # solo para pruebas automáticas
    MODO_RAPIDO = True

# El conjunto de prueba se usa UNA sola vez (semana 13). Déjenlo en False hasta entonces.
USAR_PRUEBA = False

# Dónde se guarda todo. "trabajo" es la corrida oficial (la del repositorio).
# Para los experimentos de la guía de dominio usen "trabajo_pruebas": así nunca
# se pisan los resultados oficiales (ver docs/00_como_ejecutar.md, sección 7).
CARPETA_TRABAJO = "trabajo"

EN_COLAB = "google.colab" in sys.modules
if EN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    # Carpeta del repositorio dentro de Drive (ver docs/00_como_ejecutar.md).
    RAIZ = Path("/content/drive/MyDrive/quickdraw-early")
else:
    RAIZ = Path(os.environ.get("QD_RAIZ", Path.cwd().parent))

sys.path.insert(0, str(RAIZ))
os.chdir(RAIZ)
print("Carpeta del proyecto:", RAIZ)

In [ ]:
from dataclasses import replace
import pandas as pd
import torch
from IPython.display import Image, display

from quickdraw.config import Config, config_rapida
from quickdraw import experimentos as E, figuras

cfg = Config(ruta_muestra=Path(os.environ.get("QD_MUESTRA", RAIZ / "datos" / "muestra_30")),
             ruta_trabajo=RAIZ / ("trabajo_rapido" if MODO_RAPIDO else CARPETA_TRABAJO),
             usar_prueba=USAR_PRUEBA)
if MODO_RAPIDO:
    cfg = config_rapida(cfg)
# Para un experimento, cambien parámetros AQUÍ (después de config_rapida, que si
# no los pisaría), por ejemplo:
# cfg = replace(cfg, dibujos_por_categoria=2000)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no hay (será lento)")
print(cfg)

## 2. Datos

Lee la muestra de 10 000 dibujos reconocidos por categoría, la divide 70/15/15 **por dibujo** y rasteriza todos los prefijos con normalización causal (`docs/01_datos.md` y `docs/02_prefijos_y_rasterizacion.md`). Si la carpeta `datos/muestra_30` no está, se descarga con el muestreo por reservorio (tarda; mejor súbanla a Drive).

In [ ]:
if not (Path(cfg.ruta_muestra) / "cat.ndjson.gz").exists():
    print("No está la muestra: se descarga recorriendo los archivos completos (unos minutos).")
    import subprocess
    subprocess.run([sys.executable, "datos/download_sample.py", str(cfg.ruta_muestra)], check=True)

D = E.preparar_datos(cfg, procesos=os.cpu_count() or 2)
display(pd.read_csv(cfg.ruta_resultados / "tabla_hechos.csv"))

In [ ]:
# Figura 1 del paper: prefijos con normalización causal frente a la caja del dibujo completo
fig = figuras.prefijos(D["dibujos"], [D["partes"]["entrenamiento"][i] for i in (0, 1)], cfg,
                       cfg.ruta_resultados / "fig_prefijos.png")

## 3. Búsqueda de hiperparámetros

Solo para M2: optimizador (Adam o SGD con momentum) × tasa de aprendizaje × dropout, con 5 épocas y el 30 % del entrenamiento. M1 prueba solo las tasas con Adam. M2-P usa la mejor combinación de M2, para que H2 compare una sola diferencia: los datos de entrenamiento (`docs/04_entrenamiento_y_busqueda.md`).

In [ ]:
hp, tabla_busqueda = E.buscar_hiperparametros(cfg, D)
display(tabla_busqueda)
print("Elegidos:", hp)

## 4. Entrenamiento

M1 (una semilla), M2 y M2-P (tres semillas cada uno). Cada modelo guarda un checkpoint por época en Drive; si ya terminó, solo se carga.

In [ ]:
modelos = E.entrenar_modelos(cfg, D, hp)
print(f"{len(modelos)} modelos listos")

## 5. Evaluación en validación

Todo lo de esta sección usa **solo validación**. Sirve para preparar el análisis y detectar problemas antes de la semana 13. Las hipótesis se confirman solo si el criterio se cumple en las tres semillas (`docs/06_estadistica_e_hipotesis.md`).

In [ ]:
preds_val = E.predecir_todo(cfg, D, modelos, "validacion")
R_val = E.evaluar(cfg, D, modelos, preds_val, "validacion")
display(R_val["agregada"])
display(R_val["veredictos"])

In [ ]:
carpeta = cfg.ruta_resultados / "validacion"
for nombre in ["fig_curvas_fraccion", "fig_exactitud_por_k", "fig_tsne", "fig_h4", "fig_h5",
               "fig_confusion_f30", "fig_confusion_f100", "fig_trazo_a_trazo"]:
    display(Image(filename=str(carpeta / f"{nombre}.png"), width=650))

In [ ]:
# Detalle de cada hipótesis, semilla por semilla
for r in R_val["hipotesis"]:
    filas = [{k: v for k, v in f.items() if not isinstance(v, list)} for f in r["por_semilla"]]
    print(r["hipotesis"], "→", r["veredicto"])
    display(pd.DataFrame(filas).round(4))

## 6. Latencia (CPU, un dibujo a la vez; meta: menos de 50 ms)

In [ ]:
display(E.latencia(modelos, resolucion=cfg.resolucion))

## 7. Única evaluación en prueba (semana 13)

Bloqueada mientras `USAR_PRUEBA = False`. En la semana 13, con los modelos ya congelados, se cambia a `True` en la primera celda y se ejecuta todo **una sola vez**. Los resultados quedan en `trabajo/resultados/prueba/`.

In [ ]:
if cfg.usar_prueba:
    preds_prueba = E.predecir_todo(cfg, D, modelos, "prueba")
    R_prueba = E.evaluar(cfg, D, modelos, preds_prueba, "prueba")
    display(R_prueba["agregada"])
    display(R_prueba["veredictos"])
else:
    print("Prueba bloqueada (USAR_PRUEBA = False).")

## 8. Duración de los dibujos (opcional)

Sostiene la premisa de H5: los dibujos reconocidos terminan antes del límite de 20 s porque la ronda acaba cuando la IA del juego acierta. Recorre el archivo *raw* de `cat` (varios cientos de MB).

In [ ]:
MEDIR_DURACION = False   # pónganlo en True una vez y guarden el resultado
if MEDIR_DURACION:
    import json
    from quickdraw.duracion import medir_duracion
    duracion = medir_duracion("cat")
    (cfg.ruta_resultados / "duracion_cat.json").write_text(json.dumps(duracion, indent=1))
    print(duracion)